In [36]:
#1)Entender os dados
import pandas as pd
import plotly.express as px

df_energia = pd.read_csv("/content/energia-eletrica-por-municipio.csv")

print(df_energia.shape)
df_energia.head(10)

df_energia.info()

print(df_energia["variavel"].unique())
print(df_energia["inf_categoria"].unique())
print(df_energia["inf_subcategoria"].unique())

print("Primeiro ano:", df_energia["ano"].min())
print("Último ano:", df_energia["ano"].max())
print("Municípios:", df_energia["no_mun"].nunique())

print(df_energia.isna().sum())
print("Duplicados:", df_energia.duplicated().sum())
print("Negativos:", (df_energia["valor"] < 0).sum())

(26007, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26007 entries, 0 to 26006
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   co_mun            26007 non-null  int64  
 1   no_mun            26007 non-null  object 
 2   ano               26007 non-null  int64  
 3   variavel          26007 non-null  object 
 4   inf_categoria     26007 non-null  object 
 5   inf_subcategoria  26007 non-null  object 
 6   valor             26007 non-null  float64
dtypes: float64(1), int64(2), object(4)
memory usage: 1.4+ MB
['Energia Elétrica']
['Consumidores' 'Consumo']
['Comercial' 'Consumo Próprio' 'Iluminação Pública' 'Industrial'
 'Poder Público' 'Residencial' 'Rural' 'Serviço Público']
Primeiro ano: 2010
Último ano: 2025
Municípios: 102
co_mun              0
no_mun              0
ano                 0
variavel            0
inf_categoria       0
inf_subcategoria    0
valor               0
dtype: int64
Duplic

In [37]:
# 2)Limpeza
import pandas as pd
df_energia = pd.read_csv("/content/energia-eletrica-por-municipio.csv")

df_energia = df_energia.drop_duplicates()
df_energia = df_energia.dropna(subset=["co_mun", "valor"])

df_energia = df_energia.rename(columns={"co_mun": "codigo_municipio", "no_mun": "municipio"})
df_energia["municipio"] = df_energia["municipio"].str.strip().str.title()

print("Formato do DataFrame principal:", df_energia.shape)

# Preenchimento correto dos termos com base na estrutura dos seus dados
nome_consumo = "Consumo"
nome_consumidores = "Consumidores"
coluna_classe = "inf_subcategoria"

# Filtrando com base na coluna 'inf_categoria'
df_consumo = df_energia[df_energia["inf_categoria"] == nome_consumo]
df_consumidores = df_energia[df_energia["inf_categoria"] == nome_consumidores]

print("Registros de consumo encontrados:", df_consumo.shape)
print("Registros de consumidores encontrados:", df_consumidores.shape)

# Negativos (refaturamento): remoção de valores negativos
df_consumo = df_consumo[df_consumo["valor"] >= 0]
df_consumidores = df_consumidores[df_consumidores["valor"] >= 0]

print("Registros de consumo após filtrar negativos:", df_consumo.shape)

Formato do DataFrame principal: (26007, 7)
Registros de consumo encontrados: (13008, 7)
Registros de consumidores encontrados: (12999, 7)
Registros de consumo após filtrar negativos: (13000, 7)


In [38]:
# 3)Análise
evolucao = df_consumo.groupby("ano")["valor"].sum()
print(evolucao)
print("Maior consumo:", evolucao.idxmax())
print("Menor consumo:", evolucao.idxmin())

ranking = df_consumo.groupby("municipio")["valor"].sum()
ranking = ranking.sort_values(ascending=False).head(10)
print(ranking)

tabela = df_consumo.pivot_table(index="municipio", columns="ano", values="valor", aggfunc="sum")

ano_inicial = tabela.columns.min()
ano_final = tabela.columns.max()

crescimento = pd.DataFrame()
crescimento["inicial"] = tabela[ano_inicial]
crescimento["final"] = tabela[ano_final]
crescimento["diferenca"] = crescimento["final"] - crescimento["inicial"]

# só calcula o percentual onde o valor inicial é maior que zero
crescimento = crescimento[crescimento["inicial"] > 0]
crescimento["variacao_pct"] = crescimento["diferenca"] / crescimento["inicial"] * 100

print(crescimento.sort_values("diferenca", ascending=False).head(10))
print(crescimento.sort_values("variacao_pct", ascending=False).head(10))

classes = df_consumo.groupby(coluna_classe)["valor"].sum()
participacao = classes / classes.sum() * 100
print(participacao.sort_values(ascending=False))

variacao_anual = evolucao.pct_change() * 100
print(variacao_anual)

consumo_mun = df_consumo.groupby(["municipio", "ano"])["valor"].sum().reset_index()
consumo_mun = consumo_mun.rename(columns={"valor": "consumo"})

consumidores_mun = df_consumidores.groupby(["municipio", "ano"])["valor"].sum().reset_index()
consumidores_mun = consumidores_mun.rename(columns={"valor": "consumidores"})

juntos = pd.merge(consumo_mun, consumidores_mun, on=["municipio", "ano"])
juntos = juntos[juntos["consumidores"] > 0]
juntos["consumo_por_consumidor"] = juntos["consumo"] / juntos["consumidores"]

ultimo_ano = juntos["ano"].max()
print(juntos[juntos["ano"] == ultimo_ano].sort_values("consumo_por_consumidor", ascending=False).head(10))

ano
2010    2.505458e+06
2011    2.654539e+06
2012    3.007751e+06
2013    3.223354e+06
2014    3.335540e+06
2015    3.390629e+06
2016    3.320806e+06
2017    3.307138e+06
2018    3.192847e+06
2019    2.927112e+06
2020    3.183665e+06
2021    3.276690e+06
2022    4.063918e+06
2023    4.403751e+06
2024    4.795447e+06
2025    4.943508e+06
Name: valor, dtype: float64
Maior consumo: 2025
Menor consumo: 2010
municipio
Maceió                   2.297449e+07
Marechal Deodoro         4.193945e+06
Arapiraca                4.091022e+06
Coruripe                 1.569593e+06
São Miguel Dos Campos    1.558290e+06
Rio Largo                1.250258e+06
Delmiro Gouveia          1.008494e+06
Palmeira Dos Índios      9.495159e+05
Penedo                   7.558566e+05
Maragogi                 6.920527e+05
Name: valor, dtype: float64
                         inicial         final      diferenca  variacao_pct
municipio                                                                  
Maceió                

In [39]:
# 4) Gráficos
dados = evolucao.reset_index()
fig = px.line(dados, x="ano", y="valor", title="Consumo total de energia por ano")
fig.show()

# Ranking de municípios (barras horizontais)
dados = ranking.reset_index()
fig = px.bar(dados, x="valor", y="municipio", orientation="h", title="10 municípios que mais consomem")
fig.update_yaxes(autorange="reversed")
fig.show()

# Classes ao longo dos anos (barras empilhadas)
dados = df_consumo.groupby(["ano", coluna_classe])["valor"].sum().reset_index()
fig = px.bar(dados, x="ano", y="valor", color=coluna_classe, title="Consumo por classe ao longo dos anos")
fig.show()

# Variação anual (barras)
dados = variacao_anual.reset_index()
dados.columns = ["ano", "variacao"]
fig = px.bar(dados, x="ano", y="variacao", title="Variação percentual anual do consumo")
fig.show()

# Consumo por consumidor (último ano)
dados = juntos[juntos["ano"] == ultimo_ano].sort_values("consumo_por_consumidor", ascending=False).head(10)
fig = px.bar(dados, x="consumo_por_consumidor", y="municipio", orientation="h", title="Consumo por consumidor (último ano)")
fig.update_yaxes(autorange="reversed")
fig.show()